[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/03_autogen.ipynb)

# Part 3 — Conversational multi-agent (AutoGen)

> **Control flow: emergent, from the conversation.** Who speaks next determines what happens next.

The pressure that produces this pattern: genuinely **different expertise** needs different system
prompts, different tools, and different incentives — and you want the disagreement on the record.

In [ ]:
# Setup -- same as 00_api_access.ipynb
import os, sys, time, json
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0" autogen-agentchat "autogen-ext[openai]"

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

MODEL = "gemini-3.1-flash-lite"

# AutoGen talks to Gemini through Gemini's OpenAI-compatible endpoint. Any model name
# starting with "gemini-" gets base_url set automatically.
#
# model_info must be passed explicitly: AutoGen's built-in registry stops at gemini-2.5,
# and its fallback for unrecognised models sets function_calling=False -- which would
# silently disable tool use rather than raise.
model_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=API_KEY,
    model_info=ModelInfo(vision=False, function_calling=True, json_output=True,
                         family="unknown", structured_output=True),
)
print(f"AutoGen client ready (model={MODEL}).")

### 3.1 The syntax

Participants are `AssistantAgent`s, each with its own system prompt. A team decides who speaks
next -- `RoundRobinGroupChat` just cycles. A termination condition ends the run.

Control flow is not written anywhere: it emerges from what the agents say.

In [ ]:
modeler = AssistantAgent(
    "modeler", model_client=model_client,
    system_message="You propose discretizations for PDE problems. Be concrete and specific: "
                   "name the method and the parameters. Two or three sentences. Respond to "
                   "objections rather than repeating yourself.")

analyst = AssistantAgent(
    "analyst", model_client=model_client,
    system_message="You are a numerical analyst. Challenge the modeler's proposal on "
                   "stability, conditioning, and convergence rate. Be specific about the "
                   "failure mode you are worried about. Do not be agreeable -- your value to "
                   "this conversation is the objection. Two or three sentences.")

critic = AssistantAgent(
    "critic", model_client=model_client,
    system_message="You judge the exchange between a modeler and a numerical analyst. If the "
                   "analyst's objection has been genuinely answered, reply with exactly "
                   "APPROVE followed by one sentence of justification. Otherwise state in one "
                   "sentence what is still unresolved. Do not say APPROVE merely because the "
                   "discussion is polite or has gone on a while.")

# APPROVE ends it; the message cap is the backstop so a stubborn critic cannot burn the
# free-tier daily budget.
team = RoundRobinGroupChat(
    [modeler, analyst, critic],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(10),
)

### 3.2 The exemplar: a design review

The calculator toy does not work here -- there is nothing to argue about in `13 * 47 + 8`.
A conversation pattern needs a question with genuine tension in it.

In [ ]:
TASK = ("Choose a spatial discretization for steady advection-diffusion at Peclet number "
        "~500 on an unstructured triangular mesh. State the method and any stabilization "
        "parameter.")

await Console(team.run_stream(task=TASK))

In [ ]:
await model_client.close()

### 3.3 When to reach for conversation — and the honest caveat

**Use it when** disagreement is the product: the roles have different tools and different
incentives, and you want an auditable argument rather than a confident paragraph.

**The caveat, stated plainly:** this is the easiest of the four patterns to fool yourself with.

- Three LLMs agreeing is **not** three experts agreeing. They share a prior, a training corpus,
  and a tendency toward agreeableness. A "critic" that approves everything has told you nothing.
- Distinct **tools** per role helps far more than distinct adjectives in the system prompt. An
  analyst who can actually *run* a stability calculation is a different thing from one instructed
  to "be skeptical."
- Watch the run above for the critic approving too early. If it does, that is the pattern's
  characteristic failure — not a bug in the prompt.

**Cheapest useful version:** two agents, one of which has a tool the other lacks.